##  02 · ABT

Monta a **ABT** (*Analytical Base Table*): uma linha por usuário e safra, com o alvo de churn e as medidas de cada bloco do MECE. Inclui a safra OOT (`flOOT = 1`), guardada para validar o modelo. O `02_eda` lê só as safras de treino.

**Entrada:** `workspace.tmw_loyalty.*` · **Saída:** tabela `workspace.tmw_loyalty.abt_churn`

In [0]:
%run ./00_setup

## 1. Montagem

A base tem **uma linha por usuário por safra** (`dtRef` = 1º dia de cada mês):

- **População:** quem teve pelo menos uma transação nos 28 dias antes da `dtRef`.
- **Alvo:** `churn = 1` se o usuário **não** teve nenhuma transação nos 28 dias a partir da `dtRef`.
- **Medidas:** calculadas só com transações **anteriores** à `dtRef` (nada do futuro entra).

### 1.1 Safras da análise

In [0]:
spark.sql(f"""
    SELECT IdCliente, IdTransacao, DtCriacao, QtdePontos, DescSistemaOrigem,
           to_date(DtCriacao)  AS dtDia,
           hour(DtCriacao)     AS hora,
           dayofweek(DtCriacao) AS diaSemana      -- 1 = domingo ... 7 = sábado
    FROM {T_TRANSACOES}
""").createOrReplaceTempView('tb_dia')

# definido no discovery: 2 meses de histórico no início e 28 dias de futuro no fim
DT_INICIO = '2024-03-01'
DT_OOT    = '2026-08-01'   # safra guardada para validar o modelo (fica fora da EDA)

safras = spark.sql(f"""
    SELECT explode(sequence(DATE'{DT_INICIO}', DATE'{DT_OOT}', interval 1 month)) AS dtRef
""")
safras.createOrReplaceTempView('tb_safras')

# checagem: a safra OOT ainda tem 28 dias de futuro nos dados?
max_dia = spark.sql('SELECT MAX(dtDia) FROM tb_dia').first()[0]
print(f'Safras: {safras.count()} (treino de {DT_INICIO} até o mês anterior a {DT_OOT} + OOT {DT_OOT})')
print(f'Último dia com dados: {max_dia} -> OOT {DT_OOT} tem 28 dias de futuro? '
      f"{pd.Timestamp(max_dia) >= pd.Timestamp(DT_OOT) + pd.Timedelta(days=28)}")

### 1.2 População e alvo

In [0]:
spark.sql("""
    WITH pop AS (
        SELECT DISTINCT s.dtRef, d.IdCliente
        FROM tb_safras s
        JOIN tb_dia d ON d.dtDia >= date_sub(s.dtRef, 28) AND d.dtDia < s.dtRef
    ),
    futuro AS (
        SELECT DISTINCT s.dtRef, d.IdCliente
        FROM tb_safras s
        JOIN tb_dia d ON d.dtDia >= s.dtRef AND d.dtDia < date_add(s.dtRef, 28)
    )
    SELECT p.dtRef, p.IdCliente,
           CASE WHEN f.IdCliente IS NULL THEN 1 ELSE 0 END AS churn
    FROM pop p
    LEFT JOIN futuro f ON p.dtRef = f.dtRef AND p.IdCliente = f.IdCliente
""").createOrReplaceTempView('tb_alvo')

# histórico de cada usuário antes de cada dtRef
spark.sql("""
    SELECT a.dtRef, a.IdCliente, a.churn,
           d.IdTransacao, d.DtCriacao, d.dtDia, d.hora, d.diaSemana, d.QtdePontos, d.DescSistemaOrigem,
           datediff(a.dtRef, d.dtDia) AS diasAtras          -- 1 = ontem
    FROM tb_alvo a
    JOIN tb_dia d ON d.IdCliente = a.IdCliente AND d.dtDia < a.dtRef
""").createOrReplaceTempView('tb_hist')

### 1.3 Medidas por usuário e safra

In [0]:
base = spark.sql(f"""
    WITH agg AS (
        SELECT dtRef, IdCliente, MAX(churn) AS churn,
            -- A. perfil
            MAX(diasAtras) AS diasDesdePrimeira,
            -- B. volume
            MIN(diasAtras) AS recencia,
            COUNT(DISTINCT CASE WHEN diasAtras <= 7  THEN dtDia END) AS diasAtivosD7,
            COUNT(DISTINCT CASE WHEN diasAtras <= 14 THEN dtDia END) AS diasAtivosD14,
            COUNT(DISTINCT CASE WHEN diasAtras <= 28 THEN dtDia END) AS diasAtivosD28,
            COUNT(DISTINCT CASE WHEN diasAtras <= 56 THEN dtDia END) AS diasAtivosD56,
            COUNT(CASE WHEN diasAtras <= 28 THEN IdTransacao END)    AS qtdTransacoesD28,
            SUM(CASE WHEN diasAtras <= 28 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS pontosGanhosD28,
            COUNT(DISTINCT dtDia) AS freqVida,
            -- C. tendência
            COUNT(DISTINCT CASE WHEN diasAtras BETWEEN 29 AND 56 THEN dtDia END) AS diasAtivosD28Anterior,
            -- E. hábito (horário e dia da semana, D28)
            COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 0  AND 5  THEN 1 END) AS qtdMadrugada,
            COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 6  AND 11 THEN 1 END) AS qtdManha,
            COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 12 AND 17 THEN 1 END) AS qtdTarde,
            COUNT(CASE WHEN diasAtras <= 28 AND hora BETWEEN 18 AND 23 THEN 1 END) AS qtdNoite,
            COUNT(DISTINCT CASE WHEN diasAtras <= 56 THEN diaSemana END) AS qtdDiasSemanaD56,
            -- G. loja
            SUM(CASE WHEN diasAtras <= 28 AND QtdePontos < 0 THEN -QtdePontos ELSE 0 END) AS pontosGastosD28,
            SUM(CASE WHEN diasAtras <= 28 THEN QtdePontos ELSE 0 END) AS saldoPontosD28,
            SUM(QtdePontos) AS saldoPontos,
            -- D. origem: transações vindas da plataforma de cursos
            COUNT(CASE WHEN diasAtras <= 28 AND DescSistemaOrigem = 'cursos' THEN 1 END) AS qtdCursosD28
        FROM tb_hist
        GROUP BY dtRef, IdCliente
    ),
    intervalos AS (          -- E. regularidade: dias entre um dia ativo e o seguinte (vida toda)
        SELECT dtRef, IdCliente,
               AVG(gap) AS avgIntervalo, STDDEV(gap) AS stdIntervalo, MAX(gap) AS maxIntervalo,
               COUNT(CASE WHEN gap > 28 THEN 1 END) AS qtdRetornos     -- vezes que ficou 28+ dias fora e voltou
        FROM (
            SELECT dtRef, IdCliente,
                   datediff(dtDia, LAG(dtDia) OVER (PARTITION BY dtRef, IdCliente ORDER BY dtDia)) AS gap
            FROM (SELECT DISTINCT dtRef, IdCliente, dtDia FROM tb_hist)
        )
        GROUP BY dtRef, IdCliente
    ),
    sessoes AS (             -- E. horas por dia ativo: da 1ª à última interação do dia (D28)
        SELECT dtRef, IdCliente, AVG(duracao) AS horasPorDiaD28
        FROM (
            SELECT dtRef, IdCliente, dtDia,
                   (unix_timestamp(MAX(DtCriacao)) - unix_timestamp(MIN(DtCriacao))) / 3600 AS duracao
            FROM tb_hist
            WHERE diasAtras <= 28
            GROUP BY dtRef, IdCliente, dtDia
        )
        GROUP BY dtRef, IdCliente
    )
    SELECT a.*, i.avgIntervalo, i.stdIntervalo, i.maxIntervalo, i.qtdRetornos, s.horasPorDiaD28,
           -- A. redes conectadas (foto atual do cadastro)
           c.flEmail + c.flTwitch + c.flYouTube AS qtdRedes   -- flBlueSky e flInstagram são constantes
    FROM agg a
    LEFT JOIN intervalos i ON a.dtRef = i.dtRef AND a.IdCliente = i.IdCliente
    LEFT JOIN sessoes s    ON a.dtRef = s.dtRef AND a.IdCliente = s.IdCliente
    LEFT JOIN {T_CLIENTES} c ON a.IdCliente = c.idCliente
""").toPandas()

# D. mix de produtos nos últimos 28 dias
produtos = spark.sql(f"""
    SELECT h.dtRef, h.IdCliente,
           COALESCE(p.DescNomeProduto, 'desconhecido') AS produto,   -- 0,19% dos itens fora do catálogo
           COUNT(DISTINCT h.IdTransacao) AS qtd                        -- distinct: transações com vários itens
    FROM tb_hist h
    LEFT JOIN {T_TRANSACAO_PRODUTO} tp ON h.IdTransacao = tp.IdTransacao
    LEFT JOIN {T_PRODUTOS} p           ON tp.IdProduto  = p.IdProduto
    WHERE h.diasAtras <= 28
    GROUP BY h.dtRef, h.IdCliente, COALESCE(p.DescNomeProduto, 'desconhecido')
""").toPandas()

print(f'Base de descoberta: {len(base):,} linhas (usuário × safra), {base.IdCliente.nunique():,} usuários')
print(f'Taxa média de churn: {base.churn.mean():.1%}')
base.head()

### 1.4 Clientes com volume atípico

O discovery mostrou um cliente com 31.244 transações (mediana: 8). Contas assim distorcem médias e podem ser bots ou a conta do próprio canal. Olhe o topo e decida quem sai **antes** das análises.

In [0]:
top = spark.sql(f"""
    SELECT IdCliente,
           COUNT(*)                                          AS transacoes,
           COUNT(DISTINCT to_date(DtCriacao))                AS dias_ativos,
           ROUND(COUNT(*) / COUNT(DISTINCT to_date(DtCriacao)), 1) AS transacoes_por_dia,
           ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2)  AS pct_do_total,
           MIN(DtCriacao) AS primeira, MAX(DtCriacao) AS ultima
    FROM {T_TRANSACOES}
    GROUP BY IdCliente
    ORDER BY transacoes DESC
    LIMIT 15
""").toPandas()
top

### 1.5 Medidas derivadas

In [0]:
# C. tendência
base['deltaDiasD28'] = base['diasAtivosD28'] - base['diasAtivosD28Anterior']
base['tendencia'] = np.select(
    [base['deltaDiasD28'] <= -2, base['deltaDiasD28'] >= 2], ['caindo', 'crescendo'], 'estável')
base['tendencia'] = pd.Categorical(base['tendencia'], ['caindo', 'estável', 'crescendo'], ordered=True)
base['razaoAtraso'] = base['recencia'] / base['avgIntervalo']      # recência ÷ ritmo próprio

# A. zona de risco dos novatos (3ª e 4ª semana de casa)
base['flNovatoRisco'] = base['diasDesdePrimeira'].between(15, 28).astype(int)

# B. intensidade: transações por dia ativo
base['transacoesPorDiaD28'] = base['qtdTransacoesD28'] / base['diasAtivosD28']

# E. regularidade e período do dia
base['cvIntervalo'] = base['stdIntervalo'] / base['avgIntervalo']
periodos = ['qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite']
base['periodoDominante'] = (base[periodos].idxmax(axis=1).str.replace('qtd', '').str.lower())

# F. posição relativa: percentil de pontos dentro da safra
base['percentilPontos'] = base.groupby('dtRef')['pontosGanhosD28'].rank(pct=True)
base['zScorePontos'] = base.groupby('dtRef')['pontosGanhosD28'].transform(
    lambda s: (s - s.mean()) / s.std())

# G. loja
base['gastouD28'] = np.where(base['pontosGastosD28'] > 0, 'gastou', 'não gastou')

# D. mix: os 5 produtos que somam 98% dos itens + outros + desconhecido
PRODUTOS_PRINCIPAIS = {
    'ChatMessage': 'chat',
    'Lista de presença': 'presenca',
    'Presença Streak': 'streak',
    'Resgatar Ponei': 'ponei',
    'Troca de Pontos StreamElements': 'troca',
}
produtos['grupo'] = produtos['produto'].map(PRODUTOS_PRINCIPAIS).fillna(
    produtos['produto'].where(produtos['produto'] == 'desconhecido', 'outros'))

pivot = produtos.pivot_table(index=['dtRef', 'IdCliente'], columns='grupo',
                             values='qtd', aggfunc='sum', fill_value=0)
share = pivot.div(pivot.sum(axis=1), axis=0).add_prefix('share_')
mix = pd.concat([share, pd.DataFrame({
    'produtoDominante': pivot.idxmax(axis=1),
    'qtdProdutosDistintos': (pivot > 0).sum(axis=1),
})], axis=1).reset_index()
base = base.drop(columns=[c for c in mix.columns if c in base.columns and c not in ('dtRef', 'IdCliente')])
base = base.merge(mix, on=['dtRef', 'IdCliente'], how='left')
base['produtoDominante'] = base['produtoDominante'].fillna('sem produto')

# D. origem cursos
base['usouCursosD28'] = np.where(base['qtdCursosD28'] > 0, 'usou cursos', 'não usou')

print(base['tendencia'].value_counts(normalize=True).round(3))

### 1.6 Presença nas lives e streak

Corte de **dia de live = 10 usuários**: no discovery, 25% dos dias têm até 7 usuários e a mediana é 64, então 10 separa dias com e sem live.

In [0]:
CORTE_LIVE = 10   # dias com >= 10 usuários = dia de live

comunidade = spark.sql(f"""
    WITH pop_dia AS (
        SELECT dtDia, COUNT(DISTINCT IdCliente) AS usuarios FROM tb_dia GROUP BY dtDia
    ),
    lives AS (          -- nº de dias de live nos 28 dias antes de cada safra
        SELECT s.dtRef, COUNT(*) AS qtdLivesD28
        FROM tb_safras s
        JOIN pop_dia p ON p.dtDia >= date_sub(s.dtRef, 28) AND p.dtDia < s.dtRef
        WHERE p.usuarios >= {CORTE_LIVE}
        GROUP BY s.dtRef
    ),
    dias_usuario AS (
        SELECT DISTINCT dtRef, IdCliente, dtDia FROM tb_hist WHERE diasAtras <= 28
    ),
    presenca AS (
        SELECT d.dtRef, d.IdCliente,
               AVG(p.usuarios) AS audienciaMedia,
               COUNT(CASE WHEN p.usuarios >= {CORTE_LIVE} THEN 1 END) / MAX(l.qtdLivesD28) AS pctLivesPresente
        FROM dias_usuario d
        JOIN pop_dia p ON d.dtDia = p.dtDia
        LEFT JOIN lives l ON d.dtRef = l.dtRef
        GROUP BY d.dtRef, d.IdCliente
    ),
    perdidas AS (       -- recência medida em lives: dias de live desde a última presença
        SELECT u.dtRef, u.IdCliente, COUNT(p.dtDia) AS livesPerdidas
        FROM (SELECT dtRef, IdCliente, MAX(dtDia) AS ultimoDia FROM tb_hist GROUP BY dtRef, IdCliente) u
        LEFT JOIN pop_dia p
          ON p.dtDia > u.ultimoDia AND p.dtDia < u.dtRef AND p.usuarios >= {CORTE_LIVE}
        GROUP BY u.dtRef, u.IdCliente
    )
    SELECT pr.*, pe.livesPerdidas
    FROM presenca pr
    LEFT JOIN perdidas pe ON pr.dtRef = pe.dtRef AND pr.IdCliente = pe.IdCliente
""").toPandas()

base = base.drop(columns=['audienciaMedia', 'pctLivesPresente', 'livesPerdidas'], errors='ignore') \
           .merge(comunidade, on=['dtRef', 'IdCliente'], how='left')
base[['audienciaMedia', 'pctLivesPresente', 'livesPerdidas']].describe().round(2)

In [0]:
fez = produtos.loc[produtos['produto'] == 'Presença Streak', ['dtRef', 'IdCliente']].drop_duplicates()
fez['fezStreak'] = 'fez streak'
base = base.drop(columns='fezStreak', errors='ignore').merge(fez, on=['dtRef', 'IdCliente'], how='left')
base['fezStreak'] = base['fezStreak'].fillna('não fez')

In [0]:
streak = spark.sql(f"""
    SELECT h.dtRef, h.IdCliente, MIN(h.diasAtras) AS diasUltimoStreak
    FROM tb_hist h
    JOIN {T_TRANSACAO_PRODUTO} tp ON h.IdTransacao = tp.IdTransacao
    JOIN {T_PRODUTOS} p          ON tp.IdProduto  = p.IdProduto
    WHERE p.DescNomeProduto = 'Presença Streak' AND h.diasAtras <= 28
    GROUP BY h.dtRef, h.IdCliente
""").toPandas()
base = base.drop(columns='diasUltimoStreak', errors='ignore') \
           .merge(streak, on=['dtRef', 'IdCliente'], how='left')   # nulo = sem streak em 28 dias -> imputar 28 no modelo

## 2. Salvar a base

In [0]:
TABELA_ABT = f'{SCHEMA}.abt_churn'

saida = base.copy()
saida['flOOT'] = (pd.to_datetime(saida['dtRef']) == pd.Timestamp(DT_OOT)).astype(int)
for c in saida.select_dtypes(include=['category', 'object']).columns:
    saida[c] = saida[c].astype(str)

spark.createDataFrame(saida).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TABELA_ABT)

display(saida.groupby('flOOT').agg(safras=('dtRef', 'nunique'), linhas=('churn', 'size'), churn=('churn', 'mean')).round(3))